# 13.4: A tiny KV cache

The following NumPy code implements a single attention head with and without a cache, checks that the outputs are identical, and times both. (This is the second suggested code lab.)


In [ ]:
import time
import numpy as np

rng = np.random.default_rng(0)
d, n_steps = 256, 512
Wq, Wk, Wv = (rng.standard_normal((d, d)) / np.sqrt(d) for _ in range(3))

def attend(q, K, V):
    s = K @ q / np.sqrt(d)
    a = np.exp(s - s.max()); a /= a.sum()
    return V.T @ a

def step_no_cache(xs):
    """Recompute keys and values for every past token at every step."""
    X = np.stack(xs)
    K, V = X @ Wk, X @ Wv
    return attend(xs[-1] @ Wq, K, V)

class KVCache:
    def __init__(self, max_len, d):
        self.K = np.empty((max_len, d)); self.V = np.empty((max_len, d)); self.n = 0
    def append(self, k, v):
        self.K[self.n], self.V[self.n] = k, v; self.n += 1
    def view(self):
        return self.K[:self.n], self.V[:self.n]

def step_with_cache(x, cache):
    """Compute k, v only for the new token; reuse everything else."""
    cache.append(x @ Wk, x @ Wv)
    K, V = cache.view()
    return attend(x @ Wq, K, V)

xs = [rng.standard_normal(d) for _ in range(n_steps)]

t0 = time.perf_counter()
out_a = [step_no_cache(xs[:t + 1]) for t in range(n_steps)]
t1 = time.perf_counter()
cache = KVCache(n_steps, d)
out_b = [step_with_cache(x, cache) for x in xs]
t2 = time.perf_counter()

assert np.allclose(out_a, out_b)          # identical outputs
print(f"no cache: {t1 - t0:.3f} s, with cache: {t2 - t1:.3f} s")
